# Gemma-3 4B Fine-tune (Colab)Unsloth • RAG MDN

In [ ]:
import sys, subprocess, os, redef run(cmd): subprocess.run(cmd, check=True)if "COLAB_" not in "".join(os.environ.keys()):    run([sys.executable, "-m", "pip", "install", "unsloth"])else:    import torch    v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")    run([sys.executable, "-m", "pip", "install", "--no-deps", "bitsandbytes", "accelerate", xformers, "peft", "trl", "triton", "cut_cross_entropy", "unsloth_zoo"])    run([sys.executable, "-m", "pip", "install", "sentencepiece", "protobuf", "datasets>=3.4.1,<4.0.0", "huggingface_hub>=0.34.0", "hf_transfer"])    run([sys.executable, "-m", "pip", "install", "--no-deps", "unsloth"])run([sys.executable, "-m", "pip", "install", "transformers==4.56.2"])run([sys.executable, "-m", "pip", "install", "--no-deps", "trl==0.22.2"])

In [ ]:
from google.colab import filesuploaded = files.upload()import json, oslocal_path = list(uploaded.keys())[0]dataset_path = '/content/' + local_path

In [ ]:
def chunk_text(input, max_len=1500):    parts = input.split('

')    chunks = []    buf = ''    for p in parts:        seg = p.strip()        if not seg:            continue        if len(buf) + len(seg) + (2 if buf else 0) <= max_len:            buf = (buf + ('

' if buf else '') + seg)            continue        if buf:            chunks.append(buf)            buf = ''        if len(seg) <= max_len:            chunks.append(seg)            continue        s = seg        while len(s) > max_len:            cut = s.rfind('.', 0, max_len)            idx = cut + 1 if cut > 200 else max_len            chunks.append(s[:idx].strip())            s = s[idx:].strip()        if s:            chunks.append(s)    if buf:        chunks.append(buf)    return chunks

In [ ]:
out_path = '/content/finetune_instructions.jsonl'items = json.load(open(dataset_path, 'r', encoding='utf-8'))f = open(out_path, 'w', encoding='utf-8')for it in items:    topic = it.get('topic') or ''    summary = it.get('summary') or ''    detail = it.get('detailed_knowledge') or ''    content = ('

').join([x for x in [summary, detail] if x])    for c in chunk_text(content, 1500):        rec = { 'instruction': 'Giải thích chi tiết: ' + topic, 'input': '', 'output': c }        f.write(json.dumps(rec, ensure_ascii=False) + '
')f.close()print(out_path)

## Unsloth Programmatic QLoRA

In [ ]:
import torchtorch.backends.cuda.matmul.allow_tf32 = Truetorch.backends.cudnn.allow_tf32 = Trueimport torch._dynamotorch.set_float32_matmul_precision('medium')from unsloth import FastLanguageModelfrom unsloth.chat_templates import get_chat_templatefrom datasets import load_datasetfrom trl import SFTTrainer, SFTConfigmodel_id = 'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_id, max_seq_length=2048, dtype=None, load_in_4bit=True)tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')model = FastLanguageModel.get_peft_model(model, r=64, target_modules=['q_proj','k_proj','v_proj','o_proj'], lora_alpha=16, lora_dropout=0.0, bias='none', use_gradient_checkpointing=True, random_state=3407, use_rslora=True, loftq_config=None)dataset = load_dataset('json', data_files='/content/finetune_instructions.jsonl', split='train')dataset = dataset.shuffle(seed=3407).select(range(min(len(dataset), 20000)))def to_chat_text(ex):    inst = ex['instruction']    inp = ex.get('input','')    out = ex['output']    user = inst + (('

' + inp) if inp else '')    convo = [      { 'role': 'user', 'content': user },      { 'role': 'assistant', 'content': out }    ]    txt = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')    return { 'text': txt }dataset = dataset.map(to_chat_text, remove_columns=dataset.column_names)config = SFTConfig(output_dir='/content/outputs/gemma3-4b-mdn-lora', dataset_text_field='text', max_seq_length=1024, packing=True, per_device_train_batch_size=1, gradient_accumulation_steps=8, num_train_epochs=1, learning_rate=2e-4, logging_steps=100, save_steps=1000, save_total_limit=2, bf16=True if torch.cuda.is_available() else False)trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset, args=config)trainer.train()trainer.save_model('/content/outputs/gemma3-4b-mdn-lora')tokenizer.save_pretrained('/content/outputs/gemma3-4b-mdn-lora')

## Inference Test

In [ ]:
from transformers import AutoModelForCausalLMfrom peft import PeftModelbase_id = 'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'base = AutoModelForCausalLM.from_pretrained(base_id, load_in_4bit=True, device_map='auto')lora = PeftModel.from_pretrained(base, '/content/outputs/gemma3-4b-mdn-lora')prompt = 'Giải thích cách thêm JavaScript vào HTML'convo = [ { 'role': 'user', 'content': prompt } ]txt = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=True).removeprefix('<bos>')inputs = tokenizer(txt, return_tensors='pt').to(lora.device)out = lora.generate(**inputs, max_new_tokens=256)print(tokenizer.decode(out[0], skip_special_tokens=True))

## Save to Drive

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import sys, subprocesssubprocess.run(['cp','-r','/content/outputs/gemma3-4b-mdn-lora','/content/drive/MyDrive/gemma3-4b-mdn-lora'], check=True)